# Step 2: Autograd

## 2.1 Ne Değişiyor?

Bu bölümde aynı Türkçe isim veri kümesini, tokenizer'ı ve MLP
mimarisini kullanacağım. Modelde 3424 öğrenilebilir parametre
bulunacak ve eğitim yine SGD ile yapılacak.

Değişen kısım gradyanların hesaplanmasıdır. Önceki bölümde geri
yayılımı her katman için elle yazmıştım. Bu bölümde Value sınıfı
ile sayılar üzerinde yapılan işlemleri kaydedeceğim.

loss.backward() bu işlemleri geriye doğru takip ederek zincir
kuralını uygulayacak ve gradyanları otomatik hesaplayacak.

In [1]:
import random
import math

docs = [
    l.strip().replace(" ", "").replace("İ", "i").replace("I", "ı").lower()
    for l in open("../data/raw_names.txt", encoding="utf-8")
        .read().strip().split("\n")
    if l.strip()
]

random.shuffle(docs)

uchars = sorted(set(''.join(docs)))
BOS = len(uchars)
vocab_size = len(uchars) + 1

In [2]:
print("İsim sayısı:", len(docs))
print("Karakter sayısı:", len(uchars))
print("BOS ID:", BOS)
print("Vocabulary boyutu:", vocab_size)

İsim sayısı: 1100
Karakter sayısı: 29
BOS ID: 29
Vocabulary boyutu: 30


## 2.2 Value Sınıfı

Autograd için normal sayılar yerine Value nesneleri kullanacağız.
Her nesne sayısal değerini, gradyanını, hangi değerlerden üretildiğini
ve bu değerlere göre yerel türevlerini saklayacak.

Bu bilgiler, hesaplama grafiğini oluşturmak ve daha sonra zincir
kuralıyla gradyanları otomatik hesaplamak için kullanılacak.

In [3]:
class Value:
    __slots__ = ('data', 'grad', '_children', '_local_grads')

    def __init__(self, data, children=(), local_grads=()):
        self.data = data
        self.grad = 0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1, 1))

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(
            self.data * other.data,
            (self, other),
            (other.data, self.data)
        )


    def __pow__(self, other):
        return Value(
            self.data ** other,
            (self,),
            (other * self.data ** (other - 1),)
        )
    def log(self):
        return Value(
            math.log(self.data),
            (self,),
            (1 / self.data,)
        )
    def exp(self):
        return Value(
            math.exp(self.data),
            (self,),
            (math.exp(self.data),)
        )
    def relu(self):
        return Value(
            max(0, self.data),
            (self,),
            (float(self.data > 0),)
        )
    def __neg__(self):
        return self * -1

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other ** -1

    def __rtruediv__(self, other):
        return other * self ** -1

## 2.3 İşlemleri Kaydetme: Toplama ve Çarpma

Value nesneleriyle yapılan her işlem yeni bir Value döndürür.
Bu nesne, işlem sonucunu, hangi girdilerden üretildiğini ve
bu girdilere göre yerel türevlerini saklar.

Toplamada yerel türevler 1 ve 1'dir.
Çarpmada ise her girdiye göre yerel türev, diğer girdinin değeridir.

In [4]:
a = Value(3.0)

print("Değer:", a.data)
print("Gradyan:", a.grad)
print("Girdiler:", a._children)
print("Yerel türevler:", a._local_grads)


b = Value(2.0)

toplam = a + b
carpim = a * b

print("Toplam:", toplam.data)
print("Toplamın girdileri:", [v.data for v in toplam._children])
print("Toplamın yerel türevleri:", toplam._local_grads)

print()

print("Çarpım:", carpim.data)
print("Çarpımın girdileri:", [v.data for v in carpim._children])
print("Çarpımın yerel türevleri:", carpim._local_grads)
print("Çarpımın gradyanı:", carpim.grad)

Değer: 3.0
Gradyan: 0
Girdiler: ()
Yerel türevler: ()
Toplam: 5.0
Toplamın girdileri: [3.0, 2.0]
Toplamın yerel türevleri: (1, 1)

Çarpım: 6.0
Çarpımın girdileri: [3.0, 2.0]
Çarpımın yerel türevleri: (2.0, 3.0)
Çarpımın gradyanı: 0


## 2.4 Diğer İşlemler ve Yazım Kolaylıkları

Value sınıfına üs alma, logaritma, üstel fonksiyon ve ReLU
işlemlerini ekliyoruz. Her işlem sonucunu ve yerel türevini saklıyor.

Negatif alma, çıkarma ve bölme işlemlerini ise toplama, çarpma
ve üs alma işlemlerinden oluşturuyoruz. Böylece bunlar için
ayrı türev hesapları yazmamız gerekmiyor.

In [6]:
a = Value(2.0)
b = Value(-3.0)

us = a ** 3
logaritma = a.log()
ustel = a.exp()

print("Üs:", us.data, "| Yerel türev:", us._local_grads)
print("Log:", logaritma.data, "| Yerel türev:", logaritma._local_grads)
print("Exp:", ustel.data, "| Yerel türev:", ustel._local_grads)

print("Pozitif ReLU:", a.relu().data)
print("Negatif ReLU:", b.relu().data)
print("Negatif ReLU yerel türevi:", b.relu()._local_grads)

print("Negatif alma:", (-a).data)
print("Çıkarma:", (a - 1).data)
print("Ters çıkarma:", (5 - a).data)
print("Bölme:", (a / 2).data)
print("Ters bölme:", (4 / a).data)
print("Sum:", sum([a, b]).data)

TypeError: unsupported operand type(s) for ** or pow(): 'Value' and 'int'